In [1]:
# ============================================================
# WEEK 3 — Deep Reinforcement Learning (DQN)
# Project 2: Travel & Hospitality — RL Dynamic Pricing
# Intern Branch: preeti-dev | Infotact Solutions
#
# Roadmap requirement:
#   "Replace the Q-table with a Neural Network (Deep Q-Network).
#    Handle exploration-exploitation trade-off using epsilon-
#    greedy strategy and implement experience replay to
#    stabilize training."
# ============================================================
 
 
# ── CELL 1: Install & Import Libraries ───────────────────
import subprocess, sys
 
def install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])
 
for lib in ["gymnasium", "torch", "numpy", "matplotlib", "seaborn", "pandas", "tqdm"]:
    install(lib)
 
import gymnasium as gym
from gymnasium import spaces
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import torch
import torch.nn as nn
import torch.optim as optim
from collections import deque
import random
import pickle
import os
import warnings
from tqdm import tqdm
warnings.filterwarnings('ignore')
 
os.makedirs('../reports', exist_ok=True)
os.makedirs('../models',  exist_ok=True)
os.makedirs('../data',    exist_ok=True)
 
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
 
print("✅ All libraries imported successfully")
print(f"   PyTorch version : {torch.__version__}")
print(f"   Device          : {device}")

✅ All libraries imported successfully
   PyTorch version : 2.8.0+cpu
   Device          : cpu


In [2]:
# ── CELL 2: Re-define Environment (standalone copy) ──────
MARKET_EVENTS = {
    0: {'name': 'Normal',          'demand_multiplier': 1.0,  'color': '#60a5fa'},
    1: {'name': 'Holiday Surge',   'demand_multiplier': 1.6,  'color': '#34d399'},
    2: {'name': 'Competitor Sale', 'demand_multiplier': 0.6,  'color': '#f87171'},
    3: {'name': 'Bad Weather',     'demand_multiplier': 0.75, 'color': '#fbbf24'},
}
EVENT_PROBS = [0.65, 0.12, 0.15, 0.08]
 
CUSTOMER_SEGMENTS = {
    'business':     {'price_sensitivity':0.3, 'booking_window':'any',   'base_probability':0.25},
    'leisure':       {'price_sensitivity':0.9, 'booking_window':'early', 'base_probability':0.50},
    'last_minute':   {'price_sensitivity':0.2, 'booking_window':'late',  'base_probability':0.25},
}
 
SAFETY_CONFIG = {
    'eco_floor_idx':1, 'eco_ceiling_idx':7,
    'biz_floor_idx':1, 'biz_ceiling_idx':7,
    'max_daily_swing':3, 'violation_penalty':25.0
}
 
class ContextualAirlinePricingEnv(gym.Env):
    metadata = {'render_modes': ['human']}
    ECO_PRICES        = [50, 100, 150, 200, 250, 300, 350, 400]
    BIZ_PRICES        = [300, 450, 600, 750, 900, 1050, 1100, 1200]
    COMPETITOR_PRICES = [80, 120, 160, 200, 240, 280, 320, 360]
 
    def __init__(self, eco_seats=40, biz_seats=10, total_days=30,
                 safety_config=SAFETY_CONFIG, render_mode=None):
        super().__init__()
        self.eco_seats_init, self.biz_seats_init = eco_seats, biz_seats
        self.total_days, self.render_mode, self.safety = total_days, render_mode, safety_config
        self.action_space = spaces.MultiDiscrete([len(self.ECO_PRICES), len(self.BIZ_PRICES)])
        self.observation_space = spaces.Box(
            low=np.array([0,0,0,0,0], dtype=np.float32),
            high=np.array([eco_seats,biz_seats,total_days,
                          len(self.COMPETITOR_PRICES)-1,len(MARKET_EVENTS)-1], dtype=np.float32),
            dtype=np.float32)
        self._init_state()
 
    def _init_state(self):
        self.eco_seats, self.biz_seats = self.eco_seats_init, self.biz_seats_init
        self.days_left = self.total_days
        self.total_revenue = self.eco_revenue = self.biz_revenue = 0.0
        self.market_event_id, self.competitor_idx = 0, 3
        self.prev_eco_idx = self.prev_biz_idx = 4
        self.safety_violations = 0
        self.history = []
 
    def _sample_market_event(self):
        return int(np.random.choice(len(MARKET_EVENTS), p=EVENT_PROBS))
 
    def _update_competitor_price(self):
        drift = np.random.choice([-1,0,0,1])
        self.competitor_idx = int(np.clip(self.competitor_idx+drift, 0, len(self.COMPETITOR_PRICES)-1))
 
    def _apply_safety_bounds(self, eco_idx, biz_idx):
        violated = False
        safe_eco = int(np.clip(eco_idx, self.safety['eco_floor_idx'], self.safety['eco_ceiling_idx']))
        safe_biz = int(np.clip(biz_idx, self.safety['biz_floor_idx'], self.safety['biz_ceiling_idx']))
        if safe_eco != eco_idx or safe_biz != biz_idx: violated = True
        max_swing = self.safety['max_daily_swing']
        if abs(safe_eco - self.prev_eco_idx) > max_swing:
            safe_eco = int(np.clip(self.prev_eco_idx + max_swing*np.sign(safe_eco-self.prev_eco_idx),
                                   0, len(self.ECO_PRICES)-1)); violated = True
        if abs(safe_biz - self.prev_biz_idx) > max_swing:
            safe_biz = int(np.clip(self.prev_biz_idx + max_swing*np.sign(safe_biz-self.prev_biz_idx),
                                   0, len(self.BIZ_PRICES)-1)); violated = True
        return safe_eco, safe_biz, violated
 
    def _segment_demand(self, price, days_left, seat_class, event_mult):
        max_price = max(self.BIZ_PRICES) if seat_class=='biz' else max(self.ECO_PRICES)
        total = 0
        for seg_name, seg in CUSTOMER_SEGMENTS.items():
            if seg['booking_window']=='late' and days_left>7: continue
            if seg['booking_window']=='early' and days_left<3: continue
            price_factor = max(0.0, 1.0 - seg['price_sensitivity']*(price/max_price))
            if seg_name=='last_minute': urgency = np.exp(-days_left/3)
            elif seg_name=='business':  urgency = 0.6+0.4*np.exp(-days_left/15)
            else:                      urgency = 1.0-0.6*np.exp(-days_left/20)
            competitor_eco = self.COMPETITOR_PRICES[self.competitor_idx]
            comp_factor = float(np.clip(1.0+0.3*(competitor_eco-price)/max_price, 0.5, 1.8))
            prob = seg['base_probability']*price_factor*urgency*comp_factor*event_mult + np.random.uniform(-0.05,0.05)
            prob = float(np.clip(prob, 0.0, 1.0))
            arrivals = np.random.randint(0,4)
            total += sum(np.random.random()<prob for _ in range(arrivals))
        return total
 
    def _get_obs(self):
        return np.array([self.eco_seats,self.biz_seats,self.days_left,
                         self.competitor_idx,self.market_event_id], dtype=np.float32)
 
    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self._init_state()
        self.market_event_id = self._sample_market_event()
        self._update_competitor_price()
        return self._get_obs(), {}
 
    def step(self, action):
        raw_eco, raw_biz = int(action[0]), int(action[1])
        eco_idx, biz_idx, violated = self._apply_safety_bounds(raw_eco, raw_biz)
        eco_price, biz_price = self.ECO_PRICES[eco_idx], self.BIZ_PRICES[biz_idx]
        event = MARKET_EVENTS[self.market_event_id]; event_mult = event['demand_multiplier']
        eco_bk = min(self._segment_demand(eco_price,self.days_left,'eco',event_mult), self.eco_seats)
        biz_bk = min(self._segment_demand(biz_price,self.days_left,'biz',event_mult), self.biz_seats)
        eco_rev, biz_rev = eco_price*eco_bk, biz_price*biz_bk
        daily_rev = eco_rev+biz_rev
        penalty = self.safety['violation_penalty'] if violated else 0.0
        if violated: self.safety_violations += 1
        reward = daily_rev - penalty
        self.eco_seats -= eco_bk; self.biz_seats -= biz_bk
        self.total_revenue += daily_rev; self.eco_revenue += eco_rev; self.biz_revenue += biz_rev
        self.days_left -= 1; self.prev_eco_idx, self.prev_biz_idx = eco_idx, biz_idx
        self.market_event_id = self._sample_market_event(); self._update_competitor_price()
        self.history.append({
            'day':self.total_days-self.days_left, 'days_left':self.days_left+1,
            'eco_price':eco_price,'biz_price':biz_price,
            'competitor_price':self.COMPETITOR_PRICES[self.competitor_idx],
            'market_event':event['name'],'eco_bookings':eco_bk,'biz_bookings':biz_bk,
            'eco_rev':eco_rev,'biz_rev':biz_rev,'daily_revenue':daily_rev,
            'safety_violated':violated,'eco_seats_left':self.eco_seats,
            'biz_seats_left':self.biz_seats,'total_revenue':self.total_revenue})
        terminated = (self.days_left==0 or (self.eco_seats==0 and self.biz_seats==0))
        return self._get_obs(), reward, terminated, False, {}
 
    def get_history_df(self):
        return pd.DataFrame(self.history)
 
print("✅ Environment re-loaded (same as Week 1/2, standalone copy)")
 

✅ Environment re-loaded (same as Week 1/2, standalone copy)


In [3]:
# ── CELL 3: Load Week 2 Results for Comparison ───────────
try:
    week2_comparison = pd.read_csv('../data/week2_strategy_comparison.csv')
    print("✅ Week 2 results loaded for comparison")
    print(week2_comparison.to_string(index=False))
except FileNotFoundError:
    print("⚠️  Week 2 results file not found — will proceed without it")
    week2_comparison = None

✅ Week 2 results loaded for comparison
           Strategy  Mean Revenue  Std Revenue  Safety Blocks/Ep  vs EMSR-b (%)
        Fixed Price       11576.0       1741.0              0.00          14.24
         Q-Learning       10487.0       1802.0             11.39           3.49
Time-Based Discount       10366.0       1792.0              0.00           2.30
             EMSR-b       10133.0       1617.0              0.10           0.00


In [4]:
# ── CELL 4: Why DQN? — Explaining the Upgrade ────────────
print("=" * 65)
print("   WHY UPGRADE FROM Q-TABLE TO DEEP Q-NETWORK (DQN)?")
print("=" * 65)
print()
print("  Q-Learning (Week 2) used a TABLE — one row per discrete")
print("  state bucket. Our state space had to be coarsely binned")
print("  (~3,840 buckets) to keep the table manageable.")
print()
print("  Problems this creates:")
print("    ✗ Coarse binning loses information (e.g. 39 vs 32 seats")
print("      left get treated identically if they fall in one bin)")
print("    ✗ Doesn't scale — a bigger state space explodes table size")
print("    ✗ No generalization — an unseen state has NO information,")
print("      even if it's very similar to a seen one")
print()
print("  DQN replaces the table with a NEURAL NETWORK that takes")
print("  the raw continuous state as input and outputs Q-values")
print("  for every action directly. This means:")
print("    ✓ No binning — full precision state representation")
print("    ✓ Generalizes — similar states produce similar Q-values")
print("      even for states never seen exactly during training")
print("    ✓ Scales to much larger/richer state spaces")
print("=" * 65)

   WHY UPGRADE FROM Q-TABLE TO DEEP Q-NETWORK (DQN)?

  Q-Learning (Week 2) used a TABLE — one row per discrete
  state bucket. Our state space had to be coarsely binned
  (~3,840 buckets) to keep the table manageable.

  Problems this creates:
    ✗ Coarse binning loses information (e.g. 39 vs 32 seats
      left get treated identically if they fall in one bin)
    ✗ Doesn't scale — a bigger state space explodes table size
    ✗ No generalization — an unseen state has NO information,
      even if it's very similar to a seen one

  DQN replaces the table with a NEURAL NETWORK that takes
  the raw continuous state as input and outputs Q-values
  for every action directly. This means:
    ✓ No binning — full precision state representation
    ✓ Generalizes — similar states produce similar Q-values
      even for states never seen exactly during training
    ✓ Scales to much larger/richer state spaces


In [5]:
# ── CELL 5: Flatten MultiDiscrete Action Space ───────────
# DQN's output layer needs a fixed number of discrete actions.
# We flatten [eco_price_idx, biz_price_idx] (8×8) into a single
# action index 0-63, same trick used for the Q-table in Week 2.
 
N_ECO_ACTIONS = len(ContextualAirlinePricingEnv.ECO_PRICES)
N_BIZ_ACTIONS = len(ContextualAirlinePricingEnv.BIZ_PRICES)
N_ACTIONS     = N_ECO_ACTIONS * N_BIZ_ACTIONS
 
def action_to_multi(action_idx):
    return action_idx // N_BIZ_ACTIONS, action_idx % N_BIZ_ACTIONS
 
def multi_to_action(eco_idx, biz_idx):
    return eco_idx * N_BIZ_ACTIONS + biz_idx
 
print(f"✅ Action space flattened: {N_ECO_ACTIONS} × {N_BIZ_ACTIONS} = {N_ACTIONS} actions")

✅ Action space flattened: 8 × 8 = 64 actions


In [6]:
# ── CELL 6: Q-Network Architecture ───────────────────────
class QNetwork(nn.Module):
    """
    Neural network that approximates Q(state, action) for all
    actions simultaneously — input is the 5D state, output is
    a vector of Q-values, one per possible (eco_price, biz_price)
    combination.
    """
    def __init__(self, state_dim=5, n_actions=N_ACTIONS, hidden=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, hidden),
            nn.ReLU(),
            nn.Linear(hidden, hidden),
            nn.ReLU(),
            nn.Linear(hidden, hidden // 2),
            nn.ReLU(),
            nn.Linear(hidden // 2, n_actions)
        )
 
    def forward(self, x):
        return self.net(x)
 
print("✅ QNetwork architecture defined")
print()
sample_net = QNetwork()
n_params = sum(p.numel() for p in sample_net.parameters())
print(f"   Input  : 5D state vector")
print(f"   Hidden : 128 → 128 → 64 (ReLU activations)")
print(f"   Output : {N_ACTIONS} Q-values (one per price combination)")
print(f"   Total trainable parameters: {n_params:,}")

✅ QNetwork architecture defined

   Input  : 5D state vector
   Hidden : 128 → 128 → 64 (ReLU activations)
   Output : 64 Q-values (one per price combination)
   Total trainable parameters: 29,696


In [ ]:
# ── CELL 7: Experience Replay Buffer ─────────────────────
# ★ Required by the project spec — stabilizes training by
# breaking the correlation between consecutive experiences.
# Instead of learning from each transition immediately (which
# creates unstable, correlated updates), we STORE experiences
# and learn from random MINI-BATCHES sampled from the buffer.
 
class ReplayBuffer:
    """
    Fixed-size circular buffer storing (state, action, reward,
    next_state, done) tuples. Random sampling breaks temporal
    correlation, which is what makes DQN training stable.
    """
    def __init__(self, capacity=50000):
        self.buffer = deque(maxlen=capacity)
 
    def push(self, state, action, reward, next_state, done):
        self.buffer.append((state, action, reward, next_state, done))
 
    def sample(self, batch_size):
        batch = random.sample(self.buffer, batch_size)
        states, actions, rewards, next_states, dones = zip(*batch)
        return (np.array(states), np.array(actions), np.array(rewards),
                np.array(next_states), np.array(dones))
 
    def __len__(self):
        return len(self.buffer)
 
print("✅ ReplayBuffer defined (capacity: 50,000 transitions)")